In [ ]:
from src.dfg_ms_plexus.training_setup import *

In [ ]:
# root = Path('F:/DATA/dfg_plexus/')
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')

# df = pd.read_csv(root / "radiomics_features___combined.csv", delimiter=";")
# df = pd.read_csv(root / "radiomics_features___combined_SA___normalized.csv", delimiter=";")
# df = pd.read_csv(root / "radiomics_features_shape_only___combined_SA___normalized_ICV.csv", delimiter="\t")
df = pd.read_csv(root / "radiomics_features___harmonized___combined___SA___normalized_ICV.csv", delimiter=";")

ft = df.drop(columns=['label', 'patID', 'DWI (0no,1yes)', 'LesionVolume', 'DiseaseDuration', 'EDSS'])
label = df['label'].astype(int)
label, class_mapping = get_labels_hc_cis_ms(label)  # get_labels_hc_ms, get_labels_full
print(f"{df.shape=}")
print(f"{ft.shape=}")
print(f"{label.shape=}")

In [ ]:
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(label),
                                     y=label)

# Create a dictionary mapping class labels to their corresponding weights
class_weight_dict = dict(zip(np.unique(label), class_weights))
class_weight_dict

In [ ]:
seeds = [0, 1, 2, 3, 4]

all_test_reports = []
all_train_reports = []
all_test_cms = []
all_train_cms = []
all_best_params = []

labels_sorted = np.sort(np.unique(label))
class_names = list(class_mapping.keys())

X_train, X_test, y_train, y_test = train_test_split(
    ft,
    label,
    test_size=0.3,
    random_state=42,
    stratify=label,
)

n_classes = np.unique(y_train).size
is_multiclass = n_classes > 2

for seed in seeds:
    print(f"\n========== Seed {seed} ==========")

    pipeline = Pipeline([
        ("var", VarianceThreshold(0.001)),
        ('scaler', StandardScaler()),
        # ('feat', SelectKBest(score_func=f_classif)),
        ('classifier', LGBMClassifier(
            random_state=seed,
            class_weight='balanced',  # class_weight_dict
            verbose=-1,
            subsample_freq=1
         ))
    ])

    param_grid = {
        # 'feat__k': [15, 20, 25],

        'classifier__n_estimators': [100, 200],

        'classifier__max_depth':  [2, 3, -1],
        'classifier__num_leaves': [3, 5, 7],


        'classifier__min_child_samples': [5, 10, 20],

        'classifier__colsample_bytree': [0.5, 0.7], # Forces it to look at random feature combinations
        'classifier__subsample': [0.5, 0.7],        # Forces it to look at random patient combinations
        'classifier__extra_trees': [True, False],  # Forces Random Forest style randomness!

        'classifier__learning_rate': [0.01, 0.05],
    }

    # sample_weights = compute_sample_weight(class_weight='balanced', y=label)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scorer = make_scorer(f1_score, average='macro')

    grid_search = GridSearchCV(estimator=pipeline,
                               param_grid=param_grid,
                               cv=cv,
                               scoring=scorer,
                               n_jobs=12,
                               verbose=1,
                               refit=True)

    grid_search.fit(X_train, y_train)

    import os
    import skops.io as sio

    os.makedirs('results', exist_ok=True)
    obj = sio.dump(grid_search, f'results/lgbm_clf___seed_{seed}___gridsearch.skops')

    all_best_params.append(grid_search.best_params_)

    y_train_pred = grid_search.predict(X_train)
    y_test_pred = grid_search.predict(X_test)

    train_report = classification_report(
        y_train,
        y_train_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    test_report = classification_report(
        y_test,
        y_test_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    all_train_reports.append(report_to_df(train_report))
    all_test_reports.append(report_to_df(test_report))

    all_train_cms.append(
        confusion_matrix(y_train, y_train_pred, labels=labels_sorted)
    )
    all_test_cms.append(
        confusion_matrix(y_test, y_test_pred, labels=labels_sorted)
    )

In [ ]:
train_summary = aggregate_reports(all_train_reports)
test_summary = aggregate_reports(all_test_reports)

print("\n--- Train Set: Mean ﷿﷿ Std over seeds ---")
display(train_summary)

print("\n--- Test Set: Mean ﷿﷿ Std over seeds ---")
display(test_summary)

In [ ]:
train_cms = np.asarray(all_train_cms)
test_cms = np.asarray(all_test_cms)

train_cms_norm = np.asarray([normalize_cm(cm) for cm in train_cms])
test_cms_norm = np.asarray([normalize_cm(cm) for cm in test_cms])

train_cm_mean = train_cms_norm.mean(axis=0)
train_cm_std = train_cms_norm.std(axis=0)

test_cm_mean = test_cms_norm.mean(axis=0)
test_cm_std = test_cms_norm.std(axis=0)

plot_mean_std_cm(
    train_cm_mean,
    train_cm_std,
    "Trainset Confusion Matrix: Mean ﷿﷿ Std over seeds",
    class_names,
)

plot_mean_std_cm(
    test_cm_mean,
    test_cm_std,
    "Testset Confusion Matrix: Mean ﷿﷿ Std over seeds",
    class_names,
)

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import shap

# 1. Extract the best pipeline from your XGBoost GridSearch
best_pipeline = grid_search.best_estimator_

# 2. Isolate the preprocessing steps and the trained XGBoost model
var_thresh = best_pipeline.named_steps['var']
scaler = best_pipeline.named_steps['scaler']
xgb_model = best_pipeline.named_steps['classifier']

# 3. Push the training data through the preprocessing steps
# SHAP needs the data exactly as the model saw it (scaled and variance-filtered)
X_train_var = var_thresh.transform(X_train)
X_train_scaled = scaler.transform(X_train_var)

# 4. Recover the feature names that survived the VarianceThreshold
# Assuming 'ft' is your original pandas DataFrame
surviving_features = ft.columns[var_thresh.get_support()]

# Wrap the transformed data back into a DataFrame so the SHAP plot has the actual clinical names
X_train_shap = pd.DataFrame(X_train_scaled, columns=surviving_features)

# 5. Initialize the TreeExplainer
explainer = shap.TreeExplainer(xgb_model)

# 6. Calculate the SHAP values
# For multiclass XGBoost, this returns a list of array
shap_values = explainer.shap_values(X_train_shap)

# --- Step 7: Plot the Interactive Publication-Ready SHAP Plot ---

# 7a. Calculate the mean absolute SHAP values for each class
# This safely handles both 3D arrays (new SHAP) and lists (old SHAP)
if isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    # Shape is (n_samples, n_features, n_classes)
    # Average across samples (axis 0), resulting in (n_features, n_classes)
    mean_shap_matrix = np.abs(shap_values).mean(axis=0)
    # Split into a list of 1D arrays for each class
    mean_shap_values = [mean_shap_matrix[:, i] for i in range(mean_shap_matrix.shape[1])]
elif isinstance(shap_values, list):
    # Shape is a list of (n_samples, n_features) arrays
    mean_shap_values = [np.abs(sv).mean(axis=0) for sv in shap_values]
else:
    raise ValueError("Unexpected SHAP output format.")

# 7b. Create a DataFrame for easy sorting
df_shap = pd.DataFrame({
    'Feature': surviving_features,
    'Healthy Control (0)': mean_shap_values[0],
    'CIS (1)': mean_shap_values[1],
    'MS (2)': mean_shap_values[2]
})

# Calculate total importance to sort the top features
df_shap['Total_Importance'] = df_shap['Healthy Control (0)'] + df_shap['CIS (1)'] + df_shap['MS (2)']

# Keep only the top 15 features and sort ascending (so the largest is at the top of the horizontal bar chart)
df_shap = df_shap.sort_values(by='Total_Importance', ascending=True).tail(15)

# 7c. Build the Interactive Plotly Figure
fig = go.Figure()

# Define clinical colors for your classes
colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green
classes = ['Healthy Control (0)', 'CIS (1)', 'MS (2)']

# Add a stacked bar trace for each class
for i, cls in enumerate(classes):
    fig.add_trace(go.Bar(
        y=df_shap['Feature'],
        x=df_shap[cls],
        name=cls,
        orientation='h',
        marker=dict(color=colors[i])
    ))

# 7d. Formatting and Layout Tweaks
fig.update_layout(
    title=dict(
        text="Top Biomarkers for Predicting MS Disease Spectrum (SHAP)",
        font=dict(size=20),
        y=0.95
    ),
    xaxis_title="mean(|SHAP value|) (average impact on model output magnitude)",
    yaxis_title="Radiomic Feature",
    barmode='stack',       # Stacks the bars exactly like the SHAP package
    height=800,            # Plenty of room so feature names don't overlap
    width=1100,
    hovermode="y unified", # Shows the breakdown of all 3 classes simultaneously on hover
    legend=dict(
        title="Disease Spectrum Class",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    plot_bgcolor='rgba(245, 245, 245, 1)' # Light gray background for readability
)

# Add a subtle grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='white')

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import shap
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Helper: make SHAP output class-specific
# ------------------------------------------------------------

class_names = ["Healthy Control", "CIS", "MS"]

def get_class_shap_values(shap_values, class_idx, n_features):
    """
    Returns SHAP values with shape (n_samples, n_features)
    for one class, handling both old and new SHAP formats.
    """
    if isinstance(shap_values, list):
        return shap_values[class_idx]

    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        # Common new SHAP format: (n_samples, n_features, n_classes)
        if shap_values.shape[1] == n_features:
            return shap_values[:, :, class_idx]

        # Alternative format: (n_classes, n_samples, n_features)
        if shap_values.shape[2] == n_features:
            return shap_values[class_idx, :, :]

    raise ValueError(f"Unexpected SHAP shape: {np.shape(shap_values)}")


# ------------------------------------------------------------
# Use unscaled feature values for display/coloring
# SHAP values were computed on scaled data, but original values
# are easier to interpret in beeswarm/dependence plots.
# ------------------------------------------------------------

if isinstance(X_train, pd.DataFrame):
    X_train_raw_df = X_train.copy()
else:
    X_train_raw_df = pd.DataFrame(X_train, columns=ft.columns)

X_train_display = X_train_raw_df.loc[:, surviving_features].reset_index(drop=True)
X_train_shap = X_train_shap.reset_index(drop=True)

n_features = X_train_shap.shape[1]

In [ ]:
# ------------------------------------------------------------
# Beeswarm plot for one class
# ------------------------------------------------------------

class_idx = 0  # 0 = Healthy Control, 1 = CIS, 2 = MS

shap_values_class = get_class_shap_values(
    shap_values=shap_values,
    class_idx=class_idx,
    n_features=n_features
)

# Close any leftover figures from previous plotting attempts
plt.close("all")

# Important: show=False lets us modify the SHAP figure before displaying it
shap.summary_plot(
    shap_values_class,
    X_train_display,
    plot_type="dot",
    max_display=15,
    show=False
)

plt.title(f"SHAP Beeswarm Plot for {class_names[class_idx]}", fontsize=14)
plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
plt.savefig("lgbm_clf___shap_beeswarm___HC.svg", bbox_inches="tight", format="svg")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Beeswarm plot for one class
# ------------------------------------------------------------

class_idx = 1  # 0 = Healthy Control, 1 = CIS, 2 = MS

shap_values_class = get_class_shap_values(
    shap_values=shap_values,
    class_idx=class_idx,
    n_features=n_features
)

# Close any leftover figures from previous plotting attempts
plt.close("all")

# Important: show=False lets us modify the SHAP figure before displaying it
shap.summary_plot(
    shap_values_class,
    X_train_display,
    plot_type="dot",
    max_display=15,
    show=False
)

plt.title(f"SHAP Beeswarm Plot for {class_names[class_idx]}", fontsize=14)
plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
plt.savefig("lgbm_clf___shap_beeswarm___CIS.svg", bbox_inches="tight", format="svg")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Beeswarm plot for one class
# ------------------------------------------------------------

class_idx = 2  # 0 = Healthy Control, 1 = CIS, 2 = MS

shap_values_class = get_class_shap_values(
    shap_values=shap_values,
    class_idx=class_idx,
    n_features=n_features
)

# Close any leftover figures from previous plotting attempts
plt.close("all")

# Important: show=False lets us modify the SHAP figure before displaying it
shap.summary_plot(
    shap_values_class,
    X_train_display,
    plot_type="dot",
    max_display=15,
    show=False
)

plt.title(f"SHAP Beeswarm Plot for {class_names[class_idx]}", fontsize=14)
plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
plt.savefig("lgbm_clf___shap_beeswarm___MS.svg", bbox_inches="tight", format="svg")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Dependence plot for one feature and one class
# ------------------------------------------------------------

class_idx = 2  # MS
feature_to_plot = "original_gldm_DependenceEntropy"

shap_values_class = get_class_shap_values(
    shap_values=shap_values,
    class_idx=class_idx,
    n_features=n_features
)

plt.close("all")

shap.dependence_plot(
    feature_to_plot,
    shap_values_class,
    X_train_display,
    interaction_index="auto",
    show=False
)

plt.title(f"SHAP Dependence Plot: {feature_to_plot} for {class_names[class_idx]}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Three vertically stacked class-wise SHAP beeswarm plots
# Sized for a DIN A4 manuscript page
#
# Assumes these objects already exist:
#   shap_values
#   X_train_display
#   class_names
#   n_features
#   get_class_shap_values(...)
# ============================================================

import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
import shap


# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------

CLASS_INDICES = [0, 1, 2]
PANEL_LABELS = ["A", "B", "C"]

# Number of class-specific features shown in each panel
MAX_DISPLAY = 15

# Typography: use only these two sizes
SMALL_FONT_PT = 8
LARGE_FONT_PT = 12

REQUESTED_FONT = "Calibri"

# Figure sizing:
#
# "page_content":
#   Fits inside an A4 page while reserving margins and caption space.
#
# "full_a4":
#   Produces a canvas exactly 210 × 297 mm.
FIGURE_MODE = "page_content"

# A4 page margins used in page_content mode
LEFT_MARGIN_MM = 15
RIGHT_MARGIN_MM = 15
TOP_MARGIN_MM = 12
BOTTOM_MARGIN_MM = 12

# Vertical space reserved for the figure caption
CAPTION_SPACE_MM = 25

# Output files
OUTPUT_PDF = "lgbm_clf___shap_beeswarm___all_classes_A4.pdf"
OUTPUT_SVG = "lgbm_clf___shap_beeswarm___all_classes_A4.svg"


# ------------------------------------------------------------
# Resolve Calibri
# ------------------------------------------------------------

available_fonts = {
    font.name
    for font in font_manager.fontManager.ttflist
}

if REQUESTED_FONT in available_fonts:
    FONT_FAMILY = REQUESTED_FONT
else:
    # Carlito is metrically compatible with Calibri and is often available
    # on Linux systems. DejaVu Sans is the final fallback.
    if "Carlito" in available_fonts:
        FONT_FAMILY = "Carlito"
    else:
        FONT_FAMILY = "DejaVu Sans"

    warnings.warn(
        f'"{REQUESTED_FONT}" was not found by Matplotlib. '
        f'Using "{FONT_FAMILY}" instead. Install Calibri on the system '
        "and restart Python to obtain true Calibri output."
    )


# ------------------------------------------------------------
# Global Matplotlib styling
# ------------------------------------------------------------

mpl.rcParams.update({
    # Font
    "font.family": "sans-serif",
    "font.sans-serif": [
        FONT_FAMILY,
        "Calibri",
        "Carlito",
        "Arial",
        "DejaVu Sans"
    ],

    # Only 8 pt and 12 pt text
    "font.size": SMALL_FONT_PT,
    "axes.labelsize": SMALL_FONT_PT,
    "axes.titlesize": LARGE_FONT_PT,
    "xtick.labelsize": SMALL_FONT_PT,
    "ytick.labelsize": SMALL_FONT_PT,
    "legend.fontsize": SMALL_FONT_PT,
    "figure.titlesize": LARGE_FONT_PT,

    # Editable SVG text
    "svg.fonttype": "none",

    # Embed TrueType fonts in PDF
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # Publication-style line widths
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.minor.width": 0.5,
    "ytick.minor.width": 0.5,

    # Keep minus signs compatible with the selected font
    "axes.unicode_minus": False,
})


# ------------------------------------------------------------
# DIN A4 dimensions
# ------------------------------------------------------------

MM_PER_INCH = 25.4

A4_WIDTH_MM = 210
A4_HEIGHT_MM = 297

LEFT_MARGIN_MM = 15
RIGHT_MARGIN_MM = 15
TOP_MARGIN_MM = 12
BOTTOM_MARGIN_MM = 12

# Reserve enough room for a multi-line caption and spacing
CAPTION_SPACE_MM = 45

FIGURE_WIDTH_MM = (
    A4_WIDTH_MM
    - LEFT_MARGIN_MM
    - RIGHT_MARGIN_MM
)

FIGURE_HEIGHT_MM = (
    A4_HEIGHT_MM
    - TOP_MARGIN_MM
    - BOTTOM_MARGIN_MM
    - CAPTION_SPACE_MM
)

FIGURE_SIZE_INCHES = (
    FIGURE_WIDTH_MM / MM_PER_INCH,
    FIGURE_HEIGHT_MM / MM_PER_INCH
)

print(
    f"Figure size: "
    f"{FIGURE_WIDTH_MM:.0f} × {FIGURE_HEIGHT_MM:.0f} mm"
)


def mm_to_inches(value_mm):
    """Convert millimetres to inches for Matplotlib."""
    return value_mm / MM_PER_INCH


if FIGURE_MODE == "full_a4":
    FIGURE_WIDTH_MM = A4_WIDTH_MM
    FIGURE_HEIGHT_MM = A4_HEIGHT_MM

elif FIGURE_MODE == "page_content":
    FIGURE_WIDTH_MM = (
        A4_WIDTH_MM
        - LEFT_MARGIN_MM
        - RIGHT_MARGIN_MM
    )

    FIGURE_HEIGHT_MM = (
        A4_HEIGHT_MM
        - TOP_MARGIN_MM
        - BOTTOM_MARGIN_MM
        - CAPTION_SPACE_MM
    )

else:
    raise ValueError(
        'FIGURE_MODE must be either "page_content" or "full_a4".'
    )


FIGURE_SIZE_INCHES = (
    mm_to_inches(FIGURE_WIDTH_MM),
    mm_to_inches(FIGURE_HEIGHT_MM)
)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def shap_to_array(values):
    """Convert NumPy arrays or shap.Explanation objects to an array."""
    if hasattr(values, "values"):
        array = np.asarray(values.values)
    else:
        array = np.asarray(values)

    if array.ndim != 2:
        raise ValueError(
            "Each class-specific SHAP array must have shape "
            "(n_samples, n_features). "
            f"Received shape: {array.shape}"
        )

    return array


def clean_feature_name(name):
    """Create shorter, publication-friendly radiomics labels."""
    return (
        str(name)
        .replace("original_", "")
        .replace("firstorder_", "First order: ")
        .replace("glcm_", "GLCM: ")
        .replace("gldm_", "GLDM: ")
        .replace("glrlm_", "GLRLM: ")
        .replace("glszm_", "GLSZM: ")
        .replace("ngtdm_", "NGTDM: ")
        .replace("shape_", "Shape: ")
    )


# ------------------------------------------------------------
# Collect class-specific SHAP values
# ------------------------------------------------------------

shap_by_class = []

for class_idx in CLASS_INDICES:
    class_shap_values = get_class_shap_values(
        shap_values=shap_values,
        class_idx=class_idx,
        n_features=n_features
    )

    shap_by_class.append(
        shap_to_array(class_shap_values)
    )


# ------------------------------------------------------------
# Check that the SHAP arrays are compatible
# ------------------------------------------------------------

reference_shape = shap_by_class[0].shape

for class_idx, class_values in zip(
    CLASS_INDICES,
    shap_by_class
):
    if class_values.shape != reference_shape:
        raise ValueError(
            "All class-specific SHAP arrays must have the same shape. "
            f"Class {class_idx} has shape {class_values.shape}, "
            f"but the expected shape is {reference_shape}."
        )


n_samples, n_shap_features = reference_shape


# ------------------------------------------------------------
# Prepare feature data and names
# ------------------------------------------------------------

if hasattr(X_train_display, "iloc"):
    # Pandas DataFrame
    X_plot = X_train_display.iloc[
        :n_samples,
        :n_shap_features
    ].copy()

    X_plot_array = X_plot.to_numpy()

    original_feature_names = list(
        X_plot.columns
    )

else:
    # NumPy-like input
    X_plot_array = np.asarray(X_train_display)[
        :n_samples,
        :n_shap_features
    ]

    original_feature_names = [
        f"Feature {feature_idx + 1}"
        for feature_idx in range(n_shap_features)
    ]


if X_plot_array.shape != reference_shape:
    raise ValueError(
        "The displayed feature matrix and SHAP arrays must have "
        "matching sample and feature dimensions. "
        f"X_plot has shape {X_plot_array.shape}; "
        f"SHAP values have shape {reference_shape}."
    )


clean_feature_names = [
    clean_feature_name(feature_name)
    for feature_name in original_feature_names
]


# ------------------------------------------------------------
# Calculate a separate feature order for each class
# ------------------------------------------------------------

class_orders = []
displayed_features_by_class = []

for class_values in shap_by_class:
    class_importance = np.nanmean(
        np.abs(class_values),
        axis=0
    )

    class_order = np.argsort(
        class_importance
    )[::-1]

    class_orders.append(class_order)

    displayed_features_by_class.append(
        class_order[:MAX_DISPLAY]
    )


# ------------------------------------------------------------
# Calculate a shared symmetric x-axis range
#
# Each class has its own features and ranking, but all panels
# retain the same SHAP-value scale.
# ------------------------------------------------------------

class_maxima = []

for class_values, displayed_features in zip(
    shap_by_class,
    displayed_features_by_class
):
    displayed_values = class_values[
        :,
        displayed_features
    ]

    class_maximum = np.nanmax(
        np.abs(displayed_values)
    )

    class_maxima.append(class_maximum)


max_abs_shap = np.nanmax(class_maxima)

if not np.isfinite(max_abs_shap) or max_abs_shap == 0:
    max_abs_shap = 1.0

# Five percent horizontal padding
x_limit = 1.05 * max_abs_shap


# ------------------------------------------------------------
# Create A4-sized three-panel figure
# ------------------------------------------------------------

plt.close("all")

fig = plt.figure(
    figsize=FIGURE_SIZE_INCHES,
    layout="constrained"
)

grid = fig.add_gridspec(
    nrows=len(CLASS_INDICES),
    ncols=2,
    width_ratios=[1.0, 0.025],
    height_ratios=[1.0] * len(CLASS_INDICES),

    # Reduce space between the three panels
    hspace=0.025,
    wspace=0.03
)


# Create axes with a shared x-axis
axes = []

for row_idx in range(len(CLASS_INDICES)):
    if row_idx == 0:
        axis = fig.add_subplot(
            grid[row_idx, 0]
        )
    else:
        axis = fig.add_subplot(
            grid[row_idx, 0],
            sharex=axes[0]
        )

    axes.append(axis)


# Shared colorbar axis spanning all rows
colorbar_axis = fig.add_subplot(
    grid[:, 1]
)


# ------------------------------------------------------------
# Plot class-wise beeswarm panels
# ------------------------------------------------------------

for panel_number, (
    axis,
    class_idx,
    class_values,
    class_order
) in enumerate(
    zip(
        axes,
        CLASS_INDICES,
        shap_by_class,
        class_orders
    )
):
    explanation = shap.Explanation(
        values=class_values,
        data=X_plot_array,
        feature_names=clean_feature_names
    )

    shap.plots.beeswarm(
        explanation,
        max_display=MAX_DISPLAY,

        # Separate class-wise feature ranking
        order=class_order,

        ax=axis,
        show=False,
        plot_size=None,

        # Use SHAP's standard low-to-high feature-value colors
        color=shap.plots.colors.red_blue,

        # One shared colorbar is added below
        color_bar=False,

        # Do not combine features into "sum of other features"
        group_remaining_features=False,

        # Point styling
        s=8,
        alpha=0.70
    )

    # Common horizontal scale
    axis.set_xlim(
        -x_limit,
        x_limit
    )

    # Remove SHAP-generated axis labels
    axis.set_xlabel("")
    axis.set_ylabel("")

    # Class heading: 12 pt
    axis.set_title(
        f"{PANEL_LABELS[panel_number]}   "
        f"{class_names[class_idx]}",
        loc="left",
        fontsize=LARGE_FONT_PT,
        fontfamily=FONT_FAMILY,
        fontweight="bold",
        pad=2
    )

    # Tick labels and feature labels: 8 pt
    axis.tick_params(
        axis="both",
        which="both",
        labelsize=SMALL_FONT_PT
    )

    for tick_label in axis.get_xticklabels():
        tick_label.set_fontsize(SMALL_FONT_PT)
        tick_label.set_fontfamily(FONT_FAMILY)

    for tick_label in axis.get_yticklabels():
        tick_label.set_fontsize(SMALL_FONT_PT)
        tick_label.set_fontfamily(FONT_FAMILY)

    # Zero-reference line
    axis.axvline(
        x=0,
        color="0.35",
        linewidth=0.8,
        zorder=0
    )

    # Remove unnecessary borders
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_visible(False)

    # Only show the x-axis ticks and labels on the final panel
    if panel_number < len(axes) - 1:
        axis.tick_params(
            axis="x",
            which="both",
            bottom=False,
            labelbottom=False
        )


# Shared x-axis title: 8 pt
axes[-1].set_xlabel(
    "SHAP value (impact on class output)",
    fontsize=SMALL_FONT_PT,
    fontfamily=FONT_FAMILY,
    labelpad=6
)


# ------------------------------------------------------------
# Shared qualitative feature-value colorbar
# ------------------------------------------------------------

scalar_mappable = mpl.cm.ScalarMappable(
    norm=mpl.colors.Normalize(
        vmin=0,
        vmax=1
    ),
    cmap=shap.plots.colors.red_blue
)

scalar_mappable.set_array([])

colorbar = fig.colorbar(
    scalar_mappable,
    cax=colorbar_axis
)

colorbar.set_ticks([0, 1])
colorbar.set_ticklabels(["Low", "High"])

colorbar.ax.tick_params(
    labelsize=SMALL_FONT_PT,
    length=0
)

for tick_label in colorbar.ax.get_yticklabels():
    tick_label.set_fontsize(SMALL_FONT_PT)
    tick_label.set_fontfamily(FONT_FAMILY)

colorbar.set_label(
    "Feature value",
    fontsize=SMALL_FONT_PT,
    fontfamily=FONT_FAMILY,
    labelpad=6
)

colorbar.outline.set_linewidth(0.5)


# ------------------------------------------------------------
# Export
#
# Do not use bbox_inches="tight":
# it would crop the canvas and alter the intended dimensions.
# ------------------------------------------------------------

fig.savefig(
    OUTPUT_PDF,
    format="pdf",
    bbox_inches=None,
    pad_inches=0
)

fig.savefig(
    OUTPUT_SVG,
    format="svg",
    bbox_inches=None,
    pad_inches=0
)

print(
    f"Figure size: "
    f"{FIGURE_WIDTH_MM:.0f} × {FIGURE_HEIGHT_MM:.0f} mm"
)

print(
    f"Font used: {FONT_FAMILY}"
)

print(
    f"Saved PDF: {OUTPUT_PDF}"
)

print(
    f"Saved SVG: {OUTPUT_SVG}"
)

plt.show()